# Reflection generation

Given a crystal and a space group, the library enumerates every
allowed hkl within a chosen resolution limit, computes d-spacings
from the reciprocal metric tensor, and converts them to 2θ for a
given incident wavelength. This notebook drives the
`generate_reflections` API from CIF input to 2θ tables.

We work through five sections. We load silicon from a CIF and
generate its reflection list at `d_min = 0.8` Å. We verify the
lowest-Q d-spacings against tabulated silicon values. We convert
those d-spacings to 2θ at the Cu Kα wavelength and compare with
the NIST SRM 640f certificate. We demonstrate the systematic
absence filter on a small set of forbidden Fd-3m reflections.
We close with a non-cubic contrast on corundum (R-3c), showing
the same API working on a hexagonal cell.

In [1]:
import numpy as np

from diffraction import Crystal, SpaceGroup, generate_reflections

## Section 1. Load silicon and generate the reflection list

We load the silicon structure from `silicon.cif` (COD 9008565,
a = 5.4307 Å, Fd-3m) and call `generate_reflections` with a
resolution limit of 0.8 Å. The resulting `ReflectionList` holds
an integer `hkl` array and a cached `d_spacings` array.

In [2]:
silicon = Crystal.from_cif("silicon.cif")
sg_si = SpaceGroup("Fd-3m")
rl_si = generate_reflections(silicon, sg_si, d_min=0.8)

first_hkl = tuple(int(x) for x in rl_si.hkl[0])
last_hkl = tuple(int(x) for x in rl_si.hkl[-1])

print(f"Number of allowed reflections (d >= 0.8 Å): {rl_si.hkl.shape[0]}")
print(f"d-spacing range: {rl_si.d_spacings.min():.4f} Å to "
      f"{rl_si.d_spacings.max():.4f} Å")
print(f"First hkl (largest d, smallest |Q|): {first_hkl}")
print(f"Last hkl (smallest d, largest |Q|):  {last_hkl}")

Number of allowed reflections (d >= 0.8 Å): 294
d-spacing range: 0.8187 Å to 3.1354 Å
First hkl (largest d, smallest |Q|): (-1, -1, -1)
Last hkl (smallest d, largest |Q|):  (6, 2, 2)


## Section 2. Verify d-spacings against tabulated silicon values

We print one representative of each of the ten lowest-Q allowed
families for Si in Fd-3m. Miller indices are the canonical
positive representatives; d-spacings are the library-computed
values at a = 5.4307 Å. The table starts with (1, 1, 1) at
d = 3.1354 Å, then (2, 2, 0) at 1.9200 Å and (3, 1, 1) at
1.6374 Å. These define the low-angle part of any Cu Kα Si
powder pattern.

In [3]:
# Ten lowest-Q allowed families for Si in Fd-3m. One representative
# per family; the full ReflectionList also contains their
# symmetry-equivalent hkl.
canonical_hkl = [
    (1, 1, 1), (2, 2, 0), (3, 1, 1), (4, 0, 0), (3, 3, 1),
    (4, 2, 2), (5, 1, 1), (4, 4, 0), (5, 3, 1), (6, 2, 0),
]

print(f"{'hkl':>10}  {'d (Å)':>10}")
print("-" * 24)
for h, k, ll in canonical_hkl:
    target = np.array([h, k, ll])
    idx = np.where(np.all(rl_si.hkl == target, axis=1))[0][0]
    d = rl_si.d_spacings[idx]
    print(f"({h:>2}{k:>3}{ll:>3})  {d:>10.4f}")

       hkl       d (Å)
------------------------
( 1  1  1)      3.1354
( 2  2  0)      1.9200
( 3  1  1)      1.6374
( 4  0  0)      1.3577
( 3  3  1)      1.2459
( 4  2  2)      1.1085
( 5  1  1)      1.0451
( 4  4  0)      0.9600
( 5  3  1)      0.9180
( 6  2  0)      0.8587


## Section 3. Convert d-spacings to 2θ at the Cu Kα wavelength

We call `ReflectionList.two_thetas(wavelength)` with the Cu Kα
reference wavelength λ = 1.5405929 Å (NIST SRM 640f certificate,
2019). We compare the first eleven allowed reflections with the
NIST SRM 640f Table 1 values. The notebook loads silicon at
a = 5.4307 Å (the COD value), while NIST Table 1 is computed at
a = 5.431144 Å (the certified value). The small positive offset
between library and NIST 2θ grows from +0.003° at (111) to
+0.023° at (533), reflecting the lattice-parameter difference
between the two sources, and is the expected behaviour.

In [4]:
two_thetas_si = rl_si.two_thetas(wavelength=1.5405929)

# NIST SRM 640f Table 1 (Cu Ka lambda=1.5405929 A at a=5.431144 A)
nist_table = {
    (1, 1, 1): 28.441,
    (2, 2, 0): 47.301,
    (3, 1, 1): 56.120,
    (4, 0, 0): 69.127,
    (3, 3, 1): 76.373,
    (4, 2, 2): 88.026,
    (5, 1, 1): 94.947,
    (4, 4, 0): 106.702,
    (5, 3, 1): 114.085,
    (6, 2, 0): 127.535,
    (5, 3, 3): 136.882,
}

print(f"{'hkl':>10}  {'d (Å)':>10}  {'2θ lib (°)':>12}  "
      f"{'2θ NIST (°)':>12}  {'offset (°)':>12}")
print("-" * 62)
for target_hkl, nist_2theta in nist_table.items():
    target = np.array(target_hkl)
    idx = np.where(np.all(rl_si.hkl == target, axis=1))[0][0]
    h, k, ll = target_hkl
    d = rl_si.d_spacings[idx]
    tt = two_thetas_si[idx]
    offset = tt - nist_2theta
    print(f"({h:>2}{k:>3}{ll:>3})  {d:>10.4f}  {tt:>12.2f}  "
          f"{nist_2theta:>12.2f}  {offset:>+12.2f}")

       hkl       d (Å)    2θ lib (°)   2θ NIST (°)    offset (°)
--------------------------------------------------------------
( 1  1  1)      3.1354         28.44         28.44         +0.00
( 2  2  0)      1.9200         47.30         47.30         +0.00
( 3  1  1)      1.6374         56.12         56.12         +0.00
( 4  0  0)      1.3577         69.13         69.13         +0.01
( 3  3  1)      1.2459         76.38         76.37         +0.01
( 4  2  2)      1.1085         88.03         88.03         +0.01
( 5  1  1)      1.0451         94.96         94.95         +0.01
( 4  4  0)      0.9600        106.71        106.70         +0.01
( 5  3  1)      0.9180        114.10        114.08         +0.01
( 6  2  0)      0.8587        127.55        127.53         +0.02
( 5  3  3)      0.8282        136.91        136.88         +0.02


## Section 4. Systematic absences in Fd-3m

The silicon diamond structure in Fd-3m has F-centering and a
d-glide, which together extinguish many reflections. The library's
`SpaceGroup.is_systematically_absent` applies a per-operator
h · W = h → h · t ∈ ℤ check. It catches F-centering violations
(odd (h, k, l) sums) and single-operator d-glide extinctions. We
confirm that four canonical forbidden reflections, (1, 0, 0),
(2, 0, 0), (4, 2, 0), and (6, 0, 0), are absent from the
generated list, and that (1, 1, 1) and (2, 2, 0) are present.

A note on (2, 2, 2): classical diamond-rule accounts extinguish
this reflection through multi-operator interference (the sum
over symmetry operators cancels). The library's per-operator
test does not detect that interference, so (2, 2, 2) appears in
the output of Phase 6. Phase 7 will compute the full structure
factor F(hkl), and |F(2, 2, 2)| = 0 will then extinguish it
naturally. We therefore pick absent-reflection demonstration
examples from the set of reflections the per-operator test
itself flags.

In [5]:
def _in_list(rl, target):
    t = np.array(target)
    return bool(np.any(np.all(rl.hkl == t, axis=1)))

absent_probes = [(1, 0, 0), (2, 0, 0), (4, 2, 0), (6, 0, 0)]
allowed_probes = [(1, 1, 1), (2, 2, 0), (3, 1, 1), (4, 0, 0)]

print("Absence filter -- expected absent:")
for hkl in absent_probes:
    present = _in_list(rl_si, hkl)
    status = "present" if present else "absent"
    print(f"  {hkl}: {status}")

print()
print("Absence filter -- expected allowed:")
for hkl in allowed_probes:
    present = _in_list(rl_si, hkl)
    status = "present" if present else "absent"
    print(f"  {hkl}: {status}")

Absence filter -- expected absent:
  (1, 0, 0): absent
  (2, 0, 0): absent
  (4, 2, 0): absent
  (6, 0, 0): absent

Absence filter -- expected allowed:
  (1, 1, 1): present
  (2, 2, 0): present
  (3, 1, 1): present
  (4, 0, 0): present


## Section 5. A non-cubic contrast: corundum (R-3c)

Corundum (Al₂O₃, COD 9000779) is trigonal with hexagonal setting,
a = 4.758 Å and c = 12.991 Å. The same `generate_reflections`
call works unchanged. The axis-box enumeration reads G*_ii
from the diagonal of the reciprocal metric, and the hexagonal
anisotropy (a ≠ c) is absorbed naturally. We generate at
`d_min = 1.0` Å and print the five lowest-Q reflections with
their 2θ at Cu Kα.

In [6]:
corundum = Crystal.from_cif("corundum.cif")
sg_cor = SpaceGroup("R-3c")
rl_cor = generate_reflections(corundum, sg_cor, d_min=1.0)
two_thetas_cor = rl_cor.two_thetas(wavelength=1.5405929)

print(f"Corundum: {rl_cor.hkl.shape[0]} allowed reflections at d >= 1.0 Å")
print()
print(f"{'hkl':>10}  {'d (Å)':>10}  {'2θ (°)':>10}")
print("-" * 34)
for hkl, d, tt in zip(
    rl_cor.hkl[:5], rl_cor.d_spacings[:5], two_thetas_cor[:5], strict=True
):
    h, k, ll = (int(x) for x in hkl)
    print(f"({h:>2}{k:>3}{ll:>3})  {d:>10.4f}  {tt:>10.2f}")

Corundum: 268 allowed reflections at d >= 1.0 Å

       hkl       d (Å)      2θ (°)
----------------------------------
(-1  0  2)      3.4808       25.57
( 0 -1 -2)      3.4808       25.57
( 0  1  2)      3.4808       25.57
( 1  0 -2)      3.4808       25.57
(-1  1 -2)      3.4808       25.57


The next notebook in the series continues the diffraction chain
by computing structure factors and intensities on this same
reflection list. The Phase 7 engine consumes `rl.hkl` and
`rl.d_spacings` directly as pure numpy arrays, and the
systematic-absence refinement for multi-operator extinctions
such as (2, 2, 2) resolves there as |F(hkl)| zeros.